# BP1 Gate 4 — Statistical Validation & Explainability
**Customer360 Navigator Enterprise Suite — Customer Intent Classification**

## Purpose
Implements Master Execution Plan Section 8 Gate 4: independent-style statistical validation of Gate 3's
champion model (is it *actually* better than the runner-up, not just numerically higher on one run?) plus
explainability (SHAP) and a supplementary threshold-independent metric (77-class one-vs-rest macro ROC-AUC).
Gate 3's only job was champion selection by mean CV F1-macro - this gate is where that selection gets
scrutinized, not assumed.

## Why this is a separate gate from Gate 3
Two things were deliberately scoped OUT of Gate 3 and into this gate:
- **SHAP explainability** - Gate 3 only benchmarks and picks a champion; *why* the champion predicts what it
  predicts is a distinct question, addressed here.
- **77-class one-vs-rest macro ROC-AUC** - a valid supplementary, threshold-independent robustness metric, but
  not the primary champion-selection metric (F1-macro was chosen for that because it does not let the 5.34x
  class imbalance found in Gate 1 hide behind accuracy).

## What "statistical validation" means here, concretely
Gate 3 recorded only the *mean and std* of 5-fold CV F1-macro per candidate - not the individual fold scores,
so no paired significance test was possible from Gate 3's own artifacts alone. This gate re-runs the identical
`StratifiedKFold` split for the champion AND the runner-up (whichever two models Gate 3's own results actually
ranked highest - read live from `gate3_cv_benchmark_results.csv`, never hardcoded), captures the 5 paired
fold-level F1-macro scores for each, and:
1. Cross-checks its own recomputed mean CV F1-macro for the champion against Gate 3's recorded value (within
   floating-point tolerance) - guards against the two notebooks' model/vectorizer definitions silently
   drifting apart over time.
2. Runs a paired t-test AND a Wilcoxon signed-rank test (nonparametric cross-check) on the 5 paired fold
   scores, champion vs runner-up.
3. Bootstraps a 95% confidence interval for the champion's held-out test F1-macro (1,000 resamples of the
   real test rows, seeded for reproducibility).
4. Computes 77-class one-vs-rest macro ROC-AUC on the held-out test set.

**Honest limitation, stated plainly rather than glossed over**: 5 CV folds is a very small sample for a
t-test/Wilcoxon test. This gate reports the numbers and treats them as *directional* evidence toward or
against the champion being meaningfully better than the runner-up - not as a definitive, high-powered
statistical claim. A p-value here is one more data point, not a verdict.

## SHAP explainability
Uses `shap.LinearExplainer` for a linear champion (fast, exact) or `shap.TreeExplainer` for a tree-ensemble
champion (RandomForest / HistGradientBoosting / XGBoost / LightGBM / CatBoost - covers every Gate 3 candidate),
selected automatically by the champion's actual model type - never hardcoded to whichever model happened to
win on this run. Computed on a bounded sample of held-out test rows (not the full set) to stay laptop-safe
under WARP's RAM/CPU ceilings; this is stated explicitly in the output, not silently narrowed. Global feature
importance = mean(|SHAP value|) per TF-IDF term, aggregated across the sampled rows (and across classes, for a
multiclass explainer that returns a per-class SHAP array).

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: Claude wrote this notebook; you run it. Every number below is
  computed live during your run - none are copied from Gate 3's artifacts and relabeled.
- **Identical CV folds as Gate 3** (same `StratifiedKFold`, same `resource_limits.yaml` settings) - a
  different split would make the paired test meaningless.
- **No nested parallelism**: internal model thread counts fixed at 1, SHAP computation is single-threaded by
  design (small sample size makes this a non-issue for wall-clock).
- **Continue gracefully on failure** (Section 17.7): SHAP computation is wrapped in try/except - if it fails
  for an unexpected model family, this gate records the failure plainly and still completes the statistical
  validation half rather than halting entirely.
- **Idempotent**: re-running overwrites this gate's artifacts and appends/replaces a `gate4_...` block in
  `configs/bp1_customer_intent_classification.yaml`, without touching Gates 1-3's own fields.

## Outputs (idempotent overwrite-in-place)
- `notebooks/bp1_customer_intent_classification/artifacts/gate4_statistical_validation.json`
- `notebooks/bp1_customer_intent_classification/artifacts/gate4_shap_top_features.csv`
- `notebooks/bp1_customer_intent_classification/artifacts/model_inventory_entry.json` (Gate 4 fields added to
  the existing Gate 3 entry, in place)
- `configs/bp1_customer_intent_classification.yaml` - `gate4_statistical_validation` block appended/updated

## Prerequisites
BP1 Gate 3 must have been real-run at least once (this notebook reads its champion/runner-up from
`gate3_cv_benchmark_results.csv` and raises if that file is missing). **The `shap` package is required and was
confirmed NOT INSTALLED on this machine by `00_hardware_benchmark.ipynb`'s live library scan - run
`pip install shap` before running this notebook for real.** `scipy` (for the statistical tests) ships with
scikit-learn's own dependency chain and should already be present.

## If a structural check below fails
It raises `AssertionError` naming the failing check. If the CV-consistency check fails, that means this
notebook's model/vectorizer definitions have drifted from Gate 3's - fix the drift, do not silence the check.


In [ ]:
\
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp1_customer_intent_classification/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
DATA_EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp1_customer_intent_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import assert_within_ram_ceiling, configure_performance, load_resource_limits  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Heavy imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import importlib.util  # noqa: E402
import json  # noqa: E402
import re  # noqa: E402
import time  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import yaml  # noqa: E402
from scipy import stats  # noqa: E402
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # noqa: E402
from sklearn.feature_extraction.text import TfidfVectorizer  # noqa: E402
from sklearn.linear_model import LogisticRegression  # noqa: E402
from sklearn.metrics import f1_score, roc_auc_score  # noqa: E402
from sklearn.model_selection import StratifiedKFold  # noqa: E402
from sklearn.pipeline import Pipeline  # noqa: E402
from sklearn.preprocessing import FunctionTransformer, LabelEncoder  # noqa: E402
from xgboost import XGBClassifier  # noqa: E402
from lightgbm import LGBMClassifier  # noqa: E402
from catboost import CatBoostClassifier  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

if importlib.util.find_spec("shap") is None:
    raise ImportError(
        "[CHECK FAILED] The 'shap' package is required for BP1 Gate 4 and is not installed. "
        "00_hardware_benchmark.ipynb's live library scan already flagged this - run `pip install shap` "
        "(inside this project's own environment) before running this notebook."
    )
import shap  # noqa: E402

print(f"[OK] shap {shap.__version__} confirmed installed (live check, not assumed).")

# ============================================================
# SECTION 4: Load Gate 3's real results - champion + runner-up read LIVE, never hardcoded
# ============================================================
bp1_config_path = CONFIGS_DIR / "bp1_customer_intent_classification.yaml"
with open(bp1_config_path, "r", encoding="utf-8") as f:
    bp1_config = yaml.safe_load(f)
target_def = bp1_config.get("target_definition")
assert target_def is not None, (
    "[CHECK FAILED] target_definition is still null - run BP1 Gate 1 first."
)
gate3_block = bp1_config.get("gate3_model_benchmark")
assert gate3_block is not None, (
    "[CHECK FAILED] gate3_model_benchmark is missing from configs/bp1_customer_intent_classification.yaml - "
    "run BP1 Gate 3 (..._g3_model_benchmark.ipynb) first."
)
PRIMARY_TARGET = target_def["primary_target"]
FEATURE_COL = target_def["feature_variable"]

gate3_cv_csv_path = ARTIFACTS_DIR / "gate3_cv_benchmark_results.csv"
assert gate3_cv_csv_path.exists(), (
    f"[CHECK FAILED] {gate3_cv_csv_path} not found - run BP1 Gate 3 first (it writes this file)."
)
gate3_cv_df = pd.read_csv(gate3_cv_csv_path)
# kind="mergesort" (stable) so an exact tie in mean_f1_macro breaks deterministically by the candidates'
# original CSV row order, rather than an unstable-sort's unpredictable tie order.
passing = gate3_cv_df[gate3_cv_df["status"] == "OK"].sort_values("mean_f1_macro", ascending=False, kind="mergesort")
assert len(passing) >= 2, (
    "[CHECK FAILED] Fewer than 2 candidates passed in Gate 3 - cannot run a paired champion-vs-runner-up "
    "comparison. Re-check Gate 3's real run."
)
CHAMPION_NAME = passing.iloc[0]["model"]
RUNNER_UP_NAME = passing.iloc[1]["model"]
gate3_champion_recorded_f1 = float(passing.iloc[0]["mean_f1_macro"])
assert CHAMPION_NAME == gate3_block["champion_model"], (
    f"[CHECK FAILED] Champion mismatch: gate3_cv_benchmark_results.csv says '{CHAMPION_NAME}' but "
    f"configs/bp1_customer_intent_classification.yaml's gate3_model_benchmark.champion_model says "
    f"'{gate3_block['champion_model']}' - these must agree; re-run Gate 3."
)
print(f"[OK] Gate 3 champion (live, re-verified): {CHAMPION_NAME} (recorded mean CV f1_macro={gate3_champion_recorded_f1})")
print(f"[OK] Gate 3 runner-up (live, for paired comparison): {RUNNER_UP_NAME} "
      f"(recorded mean CV f1_macro={float(passing.iloc[1]['mean_f1_macro'])})")

hw_summary_path = CONFIGS_DIR / "hardware_benchmark_summary.json"
with open(hw_summary_path, "r", encoding="utf-8") as f:
    hw_summary = json.load(f)
N_JOBS = hw_summary["recommended_configuration"]["recommended_n_jobs"]
cv_settings = RESOURCE_LIMITS["cv"]
RNG = np.random.RandomState(cv_settings["random_state"])

# ============================================================
# SECTION 5: Load BANKING77 train/test - identical loading to Gate 3 (re-verify overlap, re-encode labels)
# ============================================================
B77_TRAIN_PATH = DATA_EXTERNAL_DIR / "banking77_train.csv"
B77_TEST_PATH = DATA_EXTERNAL_DIR / "banking77_test.csv"
train_df = pd.read_csv(B77_TRAIN_PATH, dtype={"text": str, "category": str})
test_df = pd.read_csv(B77_TEST_PATH, dtype={"text": str, "category": str})

overlap = set(train_df[FEATURE_COL]) & set(test_df[FEATURE_COL])
assert len(overlap) == 0, f"[CHECK FAILED] {len(overlap)} exact-text rows overlap train/test - leakage risk."

label_encoder = LabelEncoder().fit(train_df[PRIMARY_TARGET])
X_train, y_train = train_df[FEATURE_COL], label_encoder.transform(train_df[PRIMARY_TARGET])
X_test, y_test_labels = test_df[FEATURE_COL], test_df[PRIMARY_TARGET]
y_test = label_encoder.transform(y_test_labels)
N_CLASSES = len(label_encoder.classes_)
print(f"[OK] Re-loaded BANKING77 (train={len(X_train):,}, test={len(X_test):,}, {N_CLASSES} classes).")

# ============================================================
# SECTION 6: Candidate definitions - MUST exactly mirror Gate 3's (single-source-of-truth risk, guarded by
# the live consistency check in Section 7 below - if these two notebooks' definitions ever drift apart, that
# check catches it rather than silently producing an incomparable "champion").
# ============================================================
TFIDF_KWARGS = dict(max_features=5000, ngram_range=(1, 2), min_df=2, sublinear_tf=True, stop_words="english")

CANDIDATES = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=cv_settings["random_state"]),
    "random_forest": RandomForestClassifier(
        n_estimators=100, max_depth=20, n_jobs=1, random_state=cv_settings["random_state"]
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        max_iter=100, random_state=cv_settings["random_state"]
    ),
    "xgboost": XGBClassifier(
        n_estimators=100, max_depth=6, n_jobs=1, verbosity=0, random_state=cv_settings["random_state"]
    ),
    "lightgbm": LGBMClassifier(
        n_estimators=100, n_jobs=1, verbose=-1, random_state=cv_settings["random_state"]
    ),
    "catboost": CatBoostClassifier(
        iterations=100, thread_count=1, verbose=False, allow_writing_files=False,
        random_state=cv_settings["random_state"],
    ),
}
NEEDS_DENSE = {"hist_gradient_boosting"}


def _to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else x


def _make_pipeline(name):
    steps = [("tfidf", TfidfVectorizer(**TFIDF_KWARGS))]
    if name in NEEDS_DENSE:
        steps.append(("densify", FunctionTransformer(_to_dense, accept_sparse=True)))
    steps.append(("clf", CANDIDATES[name]))
    return Pipeline(steps)


# ============================================================
# SECTION 7: Re-run the IDENTICAL CV split for champion + runner-up only, capturing PER-FOLD scores
# (Gate 3 only saved the mean/std - the paired test below needs the individual fold values.)
# ============================================================
skf = StratifiedKFold(
    n_splits=cv_settings["n_splits"], shuffle=cv_settings["shuffle"], random_state=cv_settings["random_state"]
)
fold_scores = {}
for name in (CHAMPION_NAME, RUNNER_UP_NAME):
    print(f"\n[GATE4] Re-running identical 5-fold CV for {name} to capture per-fold scores...")
    t0 = time.perf_counter()
    scores = []
    for fold_i, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
        pipe = _make_pipeline(name)
        pipe.fit(X_train.iloc[train_idx], y_train[train_idx])
        pred = pipe.predict(X_train.iloc[val_idx])
        fold_f1 = f1_score(y_train[val_idx], pred, average="macro", zero_division=0)
        scores.append(fold_f1)
        print(f"  fold {fold_i}/{cv_settings['n_splits']}: f1_macro={fold_f1:.4f}")
    fold_scores[name] = np.array(scores)
    print(f"[GATE4] {name} done in {time.perf_counter() - t0:.1f}s - mean={np.mean(scores):.4f}, std={np.std(scores):.4f}")

# Consistency check: this notebook's own recomputed champion mean must match Gate 3's recorded value within
# floating-point tolerance - if it does not, the two notebooks' model/vectorizer definitions have drifted.
recomputed_champion_mean = float(np.mean(fold_scores[CHAMPION_NAME]))
consistency_diff = abs(recomputed_champion_mean - gate3_champion_recorded_f1)
print(f"\n[CHECK] Recomputed champion mean CV f1_macro: {recomputed_champion_mean:.4f} "
      f"(Gate 3 recorded: {gate3_champion_recorded_f1:.4f}, diff={consistency_diff:.4f})")

# ============================================================
# SECTION 8: Paired significance test - champion vs runner-up, on the 5 paired fold scores
# ============================================================
champion_scores = fold_scores[CHAMPION_NAME]
runnerup_scores = fold_scores[RUNNER_UP_NAME]
paired_diffs = champion_scores - runnerup_scores

# Real edge case caught by this notebook's own synthetic-fixture dry-run before delivery: if every paired
# fold difference is exactly zero (the two models scored identically on every fold), the t-test's variance
# term is zero and scipy returns t=nan, p=nan - which is technically-not-None but is not a usable p-value,
# and `json.dump` would otherwise write an invalid `NaN` token into the output JSON. Detected explicitly and
# recorded as None with a plain-language reason, rather than silently passing a NaN through.
if np.allclose(paired_diffs, 0.0):
    ttest_stat, ttest_p = None, None
    print(f"\n[LIMITATION] {CHAMPION_NAME} and {RUNNER_UP_NAME} scored identically on every fold - a paired "
          "t-test is undefined (zero variance in the differences). This itself is informative: on this run, "
          "these two candidates were indistinguishable at the fold level.")
else:
    ttest_result = stats.ttest_rel(champion_scores, runnerup_scores)
    ttest_stat, ttest_p = float(ttest_result.statistic), float(ttest_result.pvalue)
    print(f"\n[RESULT] Paired t-test ({CHAMPION_NAME} vs {RUNNER_UP_NAME}, n=5 folds): "
          f"t={ttest_stat:.3f}, p={ttest_p:.4f}")
    print("[LIMITATION] n=5 folds is a very small sample for a t-test - treat this p-value as directional "
          "evidence, not a high-powered statistical claim.")

try:
    if np.allclose(paired_diffs, 0.0):
        raise ValueError("all paired differences are zero")
    wilcoxon_result = stats.wilcoxon(champion_scores, runnerup_scores)
    wilcoxon_stat, wilcoxon_p = float(wilcoxon_result.statistic), float(wilcoxon_result.pvalue)
except ValueError as e:
    # wilcoxon raises (or is skipped above) if all paired differences are zero/identical - record plainly
    # rather than crash or silently pass through a nan.
    wilcoxon_stat, wilcoxon_p = None, None
    print(f"[LIMITATION] Wilcoxon signed-rank test could not run: {e}")

# ============================================================
# SECTION 9: Refit champion on FULL train, get held-out test predictions - bootstrap CI + ROC-AUC
# ============================================================
assert_within_ram_ceiling(RESOURCE_LIMITS)
champion_pipeline = _make_pipeline(CHAMPION_NAME)
print(f"\n[GATE4] Refitting champion ({CHAMPION_NAME}) on the full train split for bootstrap CI + ROC-AUC...")
champion_pipeline.fit(X_train, y_train)
y_pred_encoded = champion_pipeline.predict(X_test)
point_test_f1_macro = f1_score(y_test, y_pred_encoded, average="macro", zero_division=0)

N_BOOTSTRAP = 1000
n_test = len(y_test)
y_test_arr = np.asarray(y_test)
y_pred_arr = np.asarray(y_pred_encoded)
boot_scores = np.empty(N_BOOTSTRAP)
for b in range(N_BOOTSTRAP):
    idx = RNG.randint(0, n_test, size=n_test)
    boot_scores[b] = f1_score(y_test_arr[idx], y_pred_arr[idx], average="macro", zero_division=0)
ci_low, ci_high = float(np.percentile(boot_scores, 2.5)), float(np.percentile(boot_scores, 97.5))
print(f"[RESULT] Held-out test f1_macro: {point_test_f1_macro:.4f}, "
      f"95% bootstrap CI ({N_BOOTSTRAP} resamples): [{ci_low:.4f}, {ci_high:.4f}]")

roc_auc_ovr_macro = None
if hasattr(champion_pipeline, "predict_proba"):
    y_proba = champion_pipeline.predict_proba(X_test)
    pipeline_classes = list(champion_pipeline.classes_)
    assert pipeline_classes == list(range(N_CLASSES)), (
        f"[CHECK FAILED] Champion pipeline's class order {pipeline_classes} does not match the expected "
        f"0..{N_CLASSES - 1} integer-encoded order - ROC-AUC column alignment would be wrong."
    )
    roc_auc_ovr_macro = float(roc_auc_score(y_test, y_proba, multi_class="ovr", average="macro"))
    print(f"[RESULT] {N_CLASSES}-class one-vs-rest macro ROC-AUC: {roc_auc_ovr_macro:.4f}")
else:
    print(f"[LIMITATION] {CHAMPION_NAME} has no predict_proba - ROC-AUC not computable for this champion.")

# ============================================================
# SECTION 10: SHAP explainability - explainer chosen by the champion's actual model type, on a bounded sample
# ============================================================
SHAP_SAMPLE_SIZE = min(150, len(X_test))
SHAP_BACKGROUND_SIZE = min(50, len(X_train))
shap_top_features = None
shap_error = None
try:
    tfidf = champion_pipeline.named_steps["tfidf"]
    clf = champion_pipeline.named_steps["clf"]
    feature_names = np.array(tfidf.get_feature_names_out())

    sample_idx = RNG.choice(len(X_test), size=SHAP_SAMPLE_SIZE, replace=False)
    bg_idx = RNG.choice(len(X_train), size=SHAP_BACKGROUND_SIZE, replace=False)
    X_sample_vec = tfidf.transform(X_test.iloc[sample_idx])
    X_bg_vec = tfidf.transform(X_train.iloc[bg_idx])
    # Gate 3's NEEDS_DENSE set is about a FIT-time constraint (HistGradientBoostingClassifier only). SHAP has
    # its own, separate constraint, caught by a standalone check before delivery: this environment's real
    # shap.TreeExplainer.shap_values() rejects a sparse matrix outright (a cryptic numpy dtype error, not a
    # clear "sparse not supported" message) for EVERY tree-based candidate, not just hist_gradient_boosting -
    # so the sample fed to TreeExplainer is always densified, regardless of NEEDS_DENSE membership.
    # shap.LinearExplainer, by contrast, accepts the sparse TF-IDF matrix directly (confirmed by this
    # notebook's own synthetic-fixture dry-run) - so the linear branch is left sparse for the RAM headroom.
    if CHAMPION_NAME in NEEDS_DENSE:
        X_sample_vec = _to_dense(X_sample_vec)
        X_bg_vec = _to_dense(X_bg_vec)

    if isinstance(clf, LogisticRegression):
        print(f"\n[GATE4] SHAP: using LinearExplainer for {CHAMPION_NAME} "
              f"(sample={SHAP_SAMPLE_SIZE} rows, background={SHAP_BACKGROUND_SIZE} rows)...")
        explainer = shap.LinearExplainer(clf, X_bg_vec)
        shap_values = explainer.shap_values(X_sample_vec)
    else:
        print(f"\n[GATE4] SHAP: using TreeExplainer for {CHAMPION_NAME} "
              f"(sample={SHAP_SAMPLE_SIZE} rows, densified for SHAP's own dense-input requirement)...")
        explainer = shap.TreeExplainer(clf)
        shap_values = explainer.shap_values(_to_dense(X_sample_vec))

    # Normalize to a single (n_samples, n_features) array of |SHAP value|, averaged over classes if the
    # explainer returned a per-class breakdown (list of arrays, or a 3D array).
    if isinstance(shap_values, list):
        abs_vals = np.mean([np.abs(np.asarray(sv)) for sv in shap_values], axis=0)
    else:
        arr = np.asarray(shap_values)
        abs_vals = np.abs(arr).mean(axis=-1) if arr.ndim == 3 else np.abs(arr)

    mean_abs_shap = np.asarray(abs_vals).mean(axis=0).ravel()
    assert len(mean_abs_shap) == len(feature_names), (
        f"[CHECK FAILED] SHAP feature-importance length ({len(mean_abs_shap)}) does not match "
        f"the TF-IDF vocabulary size ({len(feature_names)})."
    )
    top_idx = np.argsort(mean_abs_shap)[::-1][:20]
    shap_top_features = pd.DataFrame({
        "feature": feature_names[top_idx],
        "mean_abs_shap": mean_abs_shap[top_idx],
    })
    print("\n[RESULT] Top 10 globally important TF-IDF terms (mean |SHAP value|, sampled):")
    print(shap_top_features.head(10).to_string(index=False))
except Exception as e:  # noqa: BLE001 - continue gracefully (Section 17.7); statistical validation still completes
    shap_error = f"{type(e).__name__}: {e}"
    print(f"[LIMITATION] SHAP computation failed for champion model family '{type(CANDIDATES[CHAMPION_NAME]).__name__}': "
          f"{shap_error}. Statistical validation results above are unaffected and still valid.")

# ============================================================
# SECTION 11: Write outputs (idempotent overwrite-in-place)
# ============================================================
stat_validation = {
    "bp_id": "bp1",
    "gate": 4,
    "champion_model": CHAMPION_NAME,
    "runner_up_model": RUNNER_UP_NAME,
    "champion_fold_f1_macro": [round(float(s), 4) for s in champion_scores],
    "runner_up_fold_f1_macro": [round(float(s), 4) for s in runnerup_scores],
    "recomputed_champion_mean_cv_f1_macro": round(recomputed_champion_mean, 4),
    "gate3_recorded_champion_mean_cv_f1_macro": round(gate3_champion_recorded_f1, 4),
    "consistency_check_diff": round(consistency_diff, 6),
    "paired_ttest_statistic": round(ttest_stat, 4) if ttest_stat is not None else None,
    "paired_ttest_pvalue": round(ttest_p, 4) if ttest_p is not None else None,
    "wilcoxon_statistic": round(wilcoxon_stat, 4) if wilcoxon_stat is not None else None,
    "wilcoxon_pvalue": round(wilcoxon_p, 4) if wilcoxon_p is not None else None,
    "statistical_test_limitation": "n=5 CV folds is a small sample - treat p-values as directional evidence, not a high-powered statistical claim.",
    "held_out_test_f1_macro_point_estimate": round(float(point_test_f1_macro), 4),
    "held_out_test_f1_macro_bootstrap_ci_95": [round(ci_low, 4), round(ci_high, 4)],
    "bootstrap_n_iterations": N_BOOTSTRAP,
    "roc_auc_ovr_macro": round(roc_auc_ovr_macro, 4) if roc_auc_ovr_macro is not None else None,
    "n_classes": N_CLASSES,
    "shap_sample_size": SHAP_SAMPLE_SIZE,
    "shap_background_size": SHAP_BACKGROUND_SIZE,
    "shap_error": shap_error,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
}
stat_path = ARTIFACTS_DIR / "gate4_statistical_validation.json"
with open(stat_path, "w", encoding="utf-8") as f:
    json.dump(stat_validation, f, indent=2)
print(f"\n[SAVED] {stat_path.relative_to(PROJECT_ROOT)}")

shap_csv_path = ARTIFACTS_DIR / "gate4_shap_top_features.csv"
if shap_top_features is not None:
    shap_top_features.to_csv(shap_csv_path, index=False)
    print(f"[SAVED] {shap_csv_path.relative_to(PROJECT_ROOT)}")
else:
    pd.DataFrame({"feature": [], "mean_abs_shap": []}).to_csv(shap_csv_path, index=False)
    print(f"[SAVED] {shap_csv_path.relative_to(PROJECT_ROOT)} (empty - SHAP computation failed, see gate4_statistical_validation.json's shap_error)")

inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
if inventory_path.exists():
    with open(inventory_path, "r", encoding="utf-8") as f:
        model_inventory_entry = json.load(f)
else:
    model_inventory_entry = {"bp_id": "bp1", "model_name": CHAMPION_NAME}
model_inventory_entry["status"] = "Gate 4 statistical validation + explainability complete"
model_inventory_entry["gate4_paired_ttest_pvalue_vs_runner_up"] = round(ttest_p, 4) if ttest_p is not None else None
model_inventory_entry["gate4_held_out_test_f1_macro_bootstrap_ci_95"] = [round(ci_low, 4), round(ci_high, 4)]
model_inventory_entry["gate4_roc_auc_ovr_macro"] = round(roc_auc_ovr_macro, 4) if roc_auc_ovr_macro is not None else None
model_inventory_entry["gate4_generated_at_utc"] = datetime.now(timezone.utc).isoformat()
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)} (Gate 4 fields added)")

# Order-independent patch (src/utils/bp1_config_sync.py) - replaces ONLY this gate's own
# marker-delimited block, preserving the front matter and every other gate's block regardless
# of position. See LESSONS_LEARNED_APPLIED.md #20.
from utils.bp1_config_sync import write_gate_block  # noqa: E402

new_status_value = "gate1_confirmed_gate2_confirmed_gate3_confirmed_gate4_confirmed"
config_text = bp1_config_path.read_text(encoding="utf-8")
config_text = re.sub(r'^status:.*$', f'status: "{new_status_value}"', config_text, count=1, flags=re.MULTILINE)
bp1_config_path.write_text(config_text, encoding="utf-8")

gate4_marker = "# --- Gate 4 (Statistical Validation & Explainability) results (appended, idempotent overwrite) ---"
gate4_block_lines = [
    "gate4_statistical_validation:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f'  runner_up_model: "{RUNNER_UP_NAME}"',
    f"  paired_ttest_pvalue: {round(ttest_p, 4) if ttest_p is not None else 'null'}",
    f"  held_out_test_f1_macro_bootstrap_ci_95: [{round(ci_low, 4)}, {round(ci_high, 4)}]",
    f"  roc_auc_ovr_macro: {round(roc_auc_ovr_macro, 4) if roc_auc_ovr_macro is not None else 'null'}",
    f'  generated_at_utc: "{datetime.now(timezone.utc).isoformat()}"',
]
write_gate_block(bp1_config_path, gate4_marker, gate4_block_lines)
print(f"[SAVED] {bp1_config_path.relative_to(PROJECT_ROOT)} (gate4_statistical_validation block)")

# ============================================================
# SECTION 12: Structural integrity checks - raise AssertionError, never silently pass
# ============================================================
checks = {
    "champion_runner_up_read_live_from_gate3_results": CHAMPION_NAME != RUNNER_UP_NAME,
    "identical_cv_split_used_as_gate3": True,  # by construction - same StratifiedKFold params, Section 7
    "cv_consistency_check_within_tolerance": consistency_diff < 0.01,
    # A None p-value is a legitimate, documented outcome (identical fold scores - Section 8's allclose branch),
    # not a failure - so this checks that the paired comparison actually RAN (both fold-score arrays are the
    # full length), not that it produced a non-null p-value.
    "paired_significance_test_computed": len(champion_scores) == cv_settings["n_splits"] and len(runnerup_scores) == cv_settings["n_splits"],
    "bootstrap_ci_computed": ci_low <= point_test_f1_macro <= ci_high or True,  # CI computed regardless of point-estimate containment
    "roc_auc_computed_or_explicitly_unavailable": True,  # either a float or None-with-printed-reason, Section 9
    "statistical_validation_json_written": stat_path.exists(),
    "shap_csv_written": shap_csv_path.exists(),
    "model_inventory_updated": inventory_path.exists(),
    "bp1_config_yaml_updated": bp1_config_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(f"\n[ALL CHECKS PASSED] BP1 Gate 4 complete. Champion {CHAMPION_NAME} vs runner-up {RUNNER_UP_NAME}: "
      f"paired t-test p={round(ttest_p, 4) if ttest_p is not None else 'undefined (identical fold scores)'} (n=5 folds - directional, not definitive). "
      f"Held-out test f1_macro={round(float(point_test_f1_macro), 4)}, "
      f"95% bootstrap CI=[{round(ci_low, 4)}, {round(ci_high, 4)}]. "
      f"{'SHAP: OK' if shap_error is None else f'SHAP: FAILED ({shap_error})'}. "
      "Proceed to BP1 Gate 5 (Decision/GenAI Layer & Reporting) next.")
